In [23]:
import pandas as pd

from config import PATHS

counts = lambda s: s.value_counts(normalize=True).sort_index(ascending=False)

In [24]:
t = pd.read_pickle(PATHS.model_feature_qualifications_table.pickle)
t.head()

seizures                   CNN              \
metric                 p_rayleigh_bh significant   roc_auc roc_auc_met   
patient       feature                                                    
competition-1 corrcoef      0.404490       False  0.610688       False   
              acfw_D        0.000561        True  0.610688       False   
              acfw_P        0.008206        True  0.610688       False   
              var_D         0.607431       False  0.610688       False   
              var_P         0.032674        True  0.610688       False   

                                                                              \
metric                  p_rayleigh_bh significant p_perm_bh    met qualified   
patient       feature                                                          
competition-1 corrcoef  2.284097e-106        True  0.973005   True     False   
              acfw_D     1.311392e-54        True  0.001000  False     False   
              acfw_P     1.048648e-01       False  0.006999  False     False   
              var_D      4.328165e-17        True  0.219648   True     False   
              var_P      2.566058e-58        True  0.001000  False     False   

                       ensemble                                         \
metric                  roc_auc roc_auc_met  p_rayleigh_bh significant   
patient       feature                                                    
competition-1 corrcoef  0.59268       False  1.334214e-138        True   
              acfw_D    0.59268       False   5.955082e-81        True   
              acfw_P    0.59268       False   2.067303e-02        True   
              var_D     0.59268       False   2.161849e-19        True   
              var_P     0.59268       False   1.917672e-78        True   

                                                   
metric                 p_perm_bh    met qualified  
patient       feature                              
competition-1 corrcoef  0.984603   True     False  
              acfw_D    0.000750  False     False  
              acfw_P    0.002999  False     False  
              var_D     0.258179   True     False  
              var_P     0.001000  False     False

In [25]:
counts(t[('seizures', 'significant')])

(seizures, significant)
True     0.533333
False    0.466667
Name: proportion, dtype: float64

In [26]:
# Rayleigh test significant
counts(t[('CNN', 'significant')])

(CNN, significant)
True     0.944444
False    0.055556
Name: proportion, dtype: float64

In [27]:
counts(t[('ensemble', 'significant')])

(ensemble, significant)
True     0.977778
False    0.022222
Name: proportion, dtype: float64

In [28]:
# Permutation test non-significant
counts(t[('CNN', 'met')])

(CNN, met)
True     0.522222
False    0.477778
Name: proportion, dtype: float64

In [29]:
counts(t[('ensemble', 'met')])

(ensemble, met)
True     0.5
False    0.5
Name: proportion, dtype: float64

# Combine dataframe per model vertically

In [30]:
cnn = t.drop(columns=['ensemble']).copy()
ens = t.drop(columns=['CNN']).copy()
tabs = {'CNN': cnn, 'ensemble': ens}

for model, tab in tabs.items():
    tab.rename(columns={'CNN': 'model', 'ensemble': 'model'}, inplace=True)
    tab.insert(tab.columns.get_loc(('model', 'roc_auc')), ('model', 'type'), model)

d = pd.concat(tabs.values())
d

seizures                 model            \
metric                 p_rayleigh_bh significant      type   roc_auc   
patient       feature                                                  
competition-1 corrcoef      0.404490       False       CNN  0.610688   
              acfw_D        0.000561        True       CNN  0.610688   
              acfw_P        0.008206        True       CNN  0.610688   
              var_D         0.607431       False       CNN  0.610688   
              var_P         0.032674        True       CNN  0.610688   
...                              ...         ...       ...       ...   
U002-DE01-17  Delta_P       0.000000        True  ensemble  0.787532   
              Theta_P       0.000000        True  ensemble  0.787532   
              Alpha_P       0.003536        True  ensemble  0.787532   
              Beta_P        0.003299        True  ensemble  0.787532   
              Gamma_P       0.003299        True  ensemble  0.787532   

                                                                         \
metric                 roc_auc_met  p_rayleigh_bh significant p_perm_bh   
patient       feature                                                     
competition-1 corrcoef       False  2.284097e-106        True  0.973005   
              acfw_D         False   1.311392e-54        True  0.001000   
              acfw_P         False   1.048648e-01       False  0.006999   
              var_D          False   4.328165e-17        True  0.219648   
              var_P          False   2.566058e-58        True  0.001000   
...                            ...            ...         ...       ...   
U002-DE01-17  Delta_P         True   3.369798e-18        True  0.000500   
              Theta_P         True   2.310338e-29        True  0.000857   
              Alpha_P         True   4.651312e-11        True  0.243497   
              Beta_P          True   4.336590e-07        True  0.613377   
              Gamma_P         True   2.184333e-11        True  0.010998   

                                         
metric                    met qualified  
patient       feature                    
competition-1 corrcoef   True     False  
              acfw_D    False     False  
              acfw_P    False     False  
              var_D      True     False  
              var_P     False     False  
...                       ...       ...  
U002-DE01-17  Delta_P   False     False  
              Theta_P   False     False  
              Alpha_P    True      True  
              Beta_P     True      True  
              Gamma_P   False     False  

[360 rows x 10 columns]

In [31]:
# Seizures significantly phase locked
col = d[('seizures', 'significant')]
sig = d[col]
counts(col)

(seizures, significant)
True     0.533333
False    0.466667
Name: proportion, dtype: float64

In [32]:
# Model ROC AUC sufficient
col = sig[('model', 'roc_auc_met')]
roc = sig[col]
counts(col)

(model, roc_auc_met)
True     0.416667
False    0.583333
Name: proportion, dtype: float64

In [33]:
# Model FPs significantly phase locked
col = roc[('model', 'significant')]
ray = roc[col]
counts(col)

(model, significant)
True    1.0
Name: proportion, dtype: float64

In [34]:
# Permutation test passed (FPs come from the same distribution as seizures) ~ they tend to occur in the same phase
col = ray[('model', 'met')]
perm = ray[col]
counts(col)

(model, met)
True     0.0375
False    0.9625
Name: proportion, dtype: float64

# Overall Proportions

In [35]:
# Seizures significantly phase locked
counts(d[('seizures', 'significant')])

(seizures, significant)
True     0.533333
False    0.466667
Name: proportion, dtype: float64

In [36]:
# Model ROC AUC sufficient
counts(d[('model', 'roc_auc_met')])

(model, roc_auc_met)
True     0.333333
False    0.666667
Name: proportion, dtype: float64

In [37]:
# Model FPs significantly phase locked
counts(d[('model', 'significant')])

(model, significant)
True     0.961111
False    0.038889
Name: proportion, dtype: float64

In [38]:
# Permutation test passed (FPs come from the same distribution as seizures) ~ they tend to occur in the same phase
counts(d[('model', 'met')])

(model, met)
True     0.511111
False    0.488889
Name: proportion, dtype: float64

In [39]:
counts(d[('model', 'qualified')])

(model, qualified)
True     0.008333
False    0.991667
Name: proportion, dtype: float64